## Recommended Starting Hyperparameters
1. Population Size
* Typical Range: 50–500.
* Recommendation: Start with 100–200.
* Larger populations explore more, but are slower per generation.
* With your large search space, erring toward the higher end (e.g., 200) is reasonable if your fitness function is fast.

2. Number of Generations
* Typical Range: 50–500+.
* Recommendation: 100–300 generations.
* More generations allow more refinement, but diminishing returns after a point.
* You can monitor if the best fitness plateaus early.
3. Selection Method
* Tournament selection or roulette wheel selection are both good.
* Many libraries use tournament selection by default.
4. Crossover Rate
* Typical: 0.7–0.9 (i.e., 70–90% of offspring are produced by crossover).
* Recommendation: 0.8 (80%).
5. Mutation Rate
* Typical: 0.01–0.1 (1–10% mutation rate per gene).
* Recommendation: 0.05 (5% per gene).
* With 9 genes (slots), this means about half your children will have at least one mutation per generation.
6. Elitism
* Keep the best 1–2 individuals from each generation to ensure you don’t lose the best found so far.


In [1]:
import pandas as pd
from itertools import chain
from functools import reduce

In [2]:
# Hyperparameters
POPULATION_SIZE = 150
NUM_GENERATIONS = 100
MUTATION_RATE = 0.05
EXCLUSIONS = {
    "m_flags" : [3208,3080]
}
LOWER_BOUNDS = {
    # "level" : 190
}
UPPER_BOUNDS = {}

### Load

In [3]:
def load_csv(file_path):
    """Load a CSV file into a DataFrame."""
    df = pd.read_csv(file_path,index_col=0,
                     dtype={k:str for k in ["id", "nameId", "typeId", "set","effectId","characteristic"]})
    df.index = df.index.astype(str)
    return df

In [4]:
items = load_csv("data/datasets/items.csv")
item_sets = load_csv("data/datasets/item_sets.csv")
set_effects = load_csv("data/datasets/set_effects.csv")
effects = load_csv("data/datasets/effects.csv").set_index("id")
item_types = load_csv("data/datasets/item_types.csv").set_index("id").astype(str)

/var/folders/c3/9yq4kl5j7qd715s6mq5tyslm0000gp/T/ipykernel_16848/269983047.py:3: DtypeWarning: Columns (14,15,19,20,21,22,23,24,27,28,29,30,32,33,63,68) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path,index_col=0,


### Filters

In [5]:
for k,v in EXCLUSIONS.items():
    items = items[~items[k].isin(v)]

In [6]:
remove_list = []
for k,v in LOWER_BOUNDS.items():
    remove_list += items[items["type"]!="EffectInstanceDice"][items[k] < v].index.to_list()

for k,v in UPPER_BOUNDS.items():
    remove_list += items[items["type"]!="EffectInstanceDice"][items[k] > v].index.to_list()

items = items.drop(remove_list)

In [7]:
allowed_typeIds = list([str(i) for i in chain(range(1,12), range(16,22))]) + ["82"]
weapon_typeIds = list(set(items[items["type"]=="Weapons"]["typeId"].to_list()))
normal_typeIds = [i for i in allowed_typeIds if i not in weapon_typeIds and i != "9"]
extra_typeIds = ["23","151","217"]

In [8]:
EXTRA_SLOTS = 6
pools = [items[items["typeId"].isin(weapon_typeIds)].index.to_list()]\
    + [items[items["typeId"].isin([i])].index.to_list() for i in normal_typeIds]
for _ in range(2): # Add 2 slots for rings
    pools += [items[items["typeId"].isin(["9"])].index.to_list()]
for _ in range(EXTRA_SLOTS):
    pools += [items[items["typeId"].isin(extra_typeIds)].index.to_list()]
NUM_TYPES = len(pools)

print(f"Loaded {len(items)} items with {NUM_TYPES} types and {EXTRA_SLOTS} extra slots.")

Loaded 60754 items with 16 types and 6 extra slots.


## Effect logic dictionaries

In [9]:
def get_item_effects(item):
    if isinstance(item["possibleEffects"], float) or not item["possibleEffects"]:
        # If there are no effects, return an empty Series
        return pd.Series(dtype=float)
    # Get the effects of the item
    item_effects = items.loc[[i for i in item["possibleEffects"].split(",")]]
    # Filter effects to only include those that are in the effects DataFrame
    return item_effects[item_effects["effectId"].isin(effects.index)]

In [ ]:
def get_item_contribution(item):
    # Get the effects of the item
    item_effects = get_item_effects(item)
    if item_effects.empty:
        return pd.Series(dtype=float)
    # Merge with effects DataFrame to get characteristics and operators
    item_effects["characteristic"] = effects.loc[item_effects["effectId"]]["characteristic"].values
    item_effects["characteristicOperator"] = effects.loc[item_effects["effectId"]]["characteristicOperator"].apply(lambda x: 1 if x == "+" else -1).values
    # Calculate the maximum value for each effect based on diceSide and diceNum (maximum of the two)
    item_effects["effectValue"] = item_effects["characteristicOperator"] * item_effects[["diceSide", "diceNum"]].max(axis=1)
    # Return a DataFrame with the characteristic and effectValue
    return item_effects[["characteristic", "effectValue"]].set_index("characteristic")["effectValue"].groupby("characteristic").sum()

contributions = {rid: get_item_contribution(items.loc[rid]) for rid in items.index}

In [11]:
def get_set_contribution(item_set,overlap):
    if overlap == 0:
        return pd.Series(dtype=float)
    if item_set.name not in set_effects.index:
        # If the set has no effects, return an empty Series
        return pd.Series(dtype=float)
    # Get the effects of the set based on the overlap
    s_effects = set_effects.loc[item_set.name][str(overlap)]
    if pd.isna(s_effects) or not s_effects:
        # If there are no effects, return an empty Series
        return pd.Series(dtype=float)
    s_effects = s_effects.split(",")
    if "-2" in s_effects:
        # Remove the "-2" from the list of effects
        s_effects.remove("-2")
    s_effects = item_sets.loc[s_effects]
    if s_effects.empty:
        # If there are no effects, return an empty Series
        return pd.Series(dtype=float)
    # Merge with effects DataFrame to get characteristics and operators
    s_effects["characteristic"] = effects.loc[s_effects["effectId"]]["characteristic"].values
    s_effects["characteristicOperator"] = effects.loc[s_effects["effectId"]]["characteristicOperator"].apply(lambda x: 1 if x == "+" else -1).values
    # Calculate the maximum value for each effect based on diceSide and diceNum (maximum of the two)
    s_effects["effectValue"] = s_effects["characteristicOperator"] * s_effects[["diceSide", "diceNum"]].max(axis=1)
    # Return a DataFrame with the characteristic and effectValue
    return s_effects[["characteristic", "effectValue"]].set_index("characteristic")["effectValue"].groupby("characteristic").sum()


In [12]:
set_contributions = {}
for rid,row in item_sets.iterrows():
    if isinstance(row["items"], float) or not row["items"]:
        # If there are no items, return an empty Series
        continue
    set_contributions[rid] = {}
    size = len(row["items"].split(","))
    for i in range(size):
        set_contributions[rid][i+1] = get_set_contribution(row, i+1)
    

## Damage logic

In [13]:
damage_mapper = {
        "98" : "14", # Air,
        "96" : "13", # Water
        "99" : "15", # Fire
        "97" : "10", # Earth
        "100" : "10" # Neutral (by earth)
    }

bonus_damage_mapper = {
    "98" : "91", # Air,
    "96" : "90", # Water
    "99" : "89", # Fire
    "97" : "88", # Earth
    "100" : "92", # Neutral
}

In [14]:
def get_weapon_damage(weapon):
    weapon_effects = get_item_effects(weapon)
    if weapon_effects.empty:
        return pd.DataFrame(dtype=float)

    # Calculate damage
    weapon_damage = weapon_effects[weapon_effects["effectId"].isin(damage_mapper)]
    if weapon_damage.empty:
        return pd.DataFrame(dtype=float)

    return weapon_damage

In [15]:
weapon_damages = {rid: get_weapon_damage(row) \
                    for rid, row in items[items["type"]=="Weapons"].iterrows()}

In [16]:
# weapon = items.loc["2861093746004724544"]
# chromosome = random_chromosome()
# weapon = get_chromosome_weapon(chromosome)
# totals = get_totals(chromosome)
# get_damage(weapon, chromosome)
# items.loc[[pools[idx][item] for idx,item in enumerate(chromosome)]]
# get_item_effects(weapon)[["name","effectId","diceNum","diceSide"]]


## 0. Methods

In [17]:
import random
def random_chromosome():
    return [random.randint(0, len(slot_pool)-1) for slot_pool in pools]

In [18]:
# Mutation
def mutate(chromosome, mutation_rate=MUTATION_RATE):
    # Create a new chromosome by mutating some slots
    new_chromosome = chromosome[:]
    for i, slot_items in enumerate(pools):
        if random.random() < mutation_rate:
            new_chromosome[i] = random.randint(0, len(slot_items)-1)
    return new_chromosome

In [19]:
# Crossover
def crossover(parent1, parent2):
    # Perform single-point crossover
    point = random.randint(1, NUM_TYPES-2)
    return parent1[:point] + parent2[point:]

In [20]:
# # Sanity check to make sure the function works on all items
# for row,item in items.iterrows():
#     get_item_contribution(item)

In [21]:
def get_chromosome_weapon(chromosome):
    chromosome_item_ids = [items.loc[pools[idx][item]].name for idx,item in enumerate(chromosome)]
    equipped_items = items.loc[chromosome_item_ids]
    return equipped_items[equipped_items["type"]=="Weapons"].iloc[0]

In [112]:
# Fitness function
def get_totals(chromosome):
    chromosome_items = [items.loc[pools[idx][item]].name for idx,item in enumerate(chromosome)]
    # Item effects
    lst = [contributions[item] for item in chromosome_items]

    # Set effects
    item_ids = items.loc[chromosome_items]["id"].values
    associated_sets = items.loc[chromosome_items]["set"].dropna().unique()
    for set_rid,set_row in item_sets.loc[associated_sets].iterrows():
        set_items = set_row["items"].split(",")
        overlap = len(set(set_items) & set(item_ids))
        lst.append(set_contributions[set_rid][overlap])
        
    # Totals
    totals = reduce(lambda x, y: x.add(y, fill_value=0), lst).to_dict()
    return totals

In [ ]:
def get_damage(weapon, chromosome):
    totals = get_totals(chromosome)

    # Character stats
    power = totals.get("25",0)
    base_crit_chance = totals.get("18",0)
    base_crit_added_damage = totals.get("86",0)

    # Critical hit logic
    crit_bonus = weapon["criticalHitBonus"] if not pd.isna(weapon["criticalHitBonus"]) else 0
    weapon_crit_chance = weapon["criticalHitProbability"] if not pd.isna(weapon["criticalHitProbability"]) else 0
    crit_chance = min(weapon_crit_chance + base_crit_chance,100)
    expected_crit_bonus = crit_bonus * crit_chance/100 # Expected value of distribution
    expected_crit_added_damage = base_crit_added_damage * crit_chance/100 # Expected value of distribution
    # max_crit_bonus = crit_bonus # Use this for maximum possible

    # weapon_damage = get_weapon_damage(weapon)
    weapon_damage = weapon_damages[weapon.name]

    # Calculate total damage
    damage = 0
    for _,row in weapon_damage.iterrows():
        # Get the stat and base damage
        stat = damage_mapper[row["effectId"]]
        weapon_base_damage = row[["diceNum","diceSide"]].max()
        stat_base = totals.get(stat,0)

        # Get the added bonus damage from the stat
        bonus_damage_id = bonus_damage_mapper[row["effectId"]]
        added_bonus = totals.get(bonus_damage_id, 0)
        damage += (weapon_base_damage + expected_crit_bonus) * (1+(stat_base+power)/100) + expected_crit_added_damage + added_bonus
    return damage
    

In [24]:
def fitness(chromosome):
    # Penalizations
    if len(set(chromosome[-EXTRA_SLOTS:])) < EXTRA_SLOTS:
        # Some dofus or trophies are duplicated
        return 0

    # TODO: Penalize two rings
    # return get_totals(chromosome).get(18, 0)
    weapon = get_chromosome_weapon(chromosome)
    return get_damage(weapon,chromosome)

In [25]:
# Sanity check to make sure the function works on random chromosomes
for _ in range(10):
    chromosome = random_chromosome()
    assert isinstance(fitness(chromosome), float) or isinstance(fitness(chromosome), int)

## 1. Initialize population

In [26]:
population = [random_chromosome() for _ in range(POPULATION_SIZE)]

## 2. Selection

In [27]:
def select(population, k=10):
    # Select top k chromosomes
    scored = sorted(population, key=fitness, reverse=True)
    return scored[:k]

## 3. Main Loop

In [30]:
from tqdm import tqdm
for gen in tqdm(range(NUM_GENERATIONS), desc="Generations"):
    # Select best individuals
    selected = select(population, k=10)
    # Generate new population
    new_population = selected[:]
    while len(new_population) < POPULATION_SIZE:
        parent1, parent2 = random.sample(selected, 2)
        child = crossover(parent1, parent2)
        child = mutate(child)
        new_population.append(child)
    population = new_population
    # Print best fitness in this generation
    best = max(population, key=fitness)
    # print(f'Gen {gen}: Best damage = {fitness(best)}')

Generations:   1%|          | 1/100 [00:01<02:11,  1.33s/it]

Gen 0: Best damage = 1221.212


Generations:   2%|▏         | 2/100 [00:02<02:09,  1.32s/it]

Gen 1: Best damage = 1227.1


Generations:   3%|▎         | 3/100 [00:04<02:10,  1.35s/it]

...

Gen 97: Best damage = 2056.72


Generations:  99%|█████████▉| 99/100 [02:16<00:01,  1.40s/it]

Gen 98: Best damage = 2056.72


Generations: 100%|██████████| 100/100 [02:17<00:00,  1.38s/it]

Gen 99: Best damage = 2056.72


In [125]:
tmp = items.loc[[pools[idx][item] for idx,item in enumerate(best)]]
tmp

,name,type,m_flags,id,nameId,typeId,descriptionId,iconId,level,realWeight,...,effectElement,spellId,effectTriggerDuration,zoneDescr,value,diceNum,diceSide,displayZero,class,set
2861093746004724544,Espada maldita del gran Desangrador Guerrero,Weapons,24584,8993,995863,6,507581.0,6083.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Espada,NaN
2861093746004741472,Collar rhoarim,Items,24712,31761,1138774,1,1142456.0,1329.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Amuleto,2861093746001838874
2861093746004741473,Taparrabos rhoarim,Items,24712,31762,1138777,10,1142454.0,10358.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cinturón,2861093746001838874
2861093746004731252,Botas de Miauvizor,Items,24712,17578,612510,11,612415.0,11306.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bota,2861093746001838309
2861093746004731253,Casco de Miauvizor,Items,24712,17579,610029,16,610003.0,16492.0,200.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Sombrero,2861093746001838309
2861093746004727614,Capa Brigada,Items,24712,13197,242230,17,242231.0,17254.0,198.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Capa,NaN
2861093746004728051,Lokulto,Items,24712,13673,241771,18,845905.0,18091.0,20.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Mascota,NaN
2861093746004732259,Cuadrifolio,Items,24712,18700,721342,82,721344.0,82150.0,193.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Escudo,NaN
2861093746004728415,Anillo de Mizz Frizz,Items,24712,14061,422390,9,422391.0,9267.0,198.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Anillo,2861093746001838231
2861093746004726812,Anillo Gwen Astarde,Items,24712,12100,32049,9,32050.0,9223.0,175.0,10.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Anillo,2861093746001838179


In [41]:
best

[482, 320, 350, 310, 297, 184, 89, 119, 264, 226, 5, 265, 95, 107, 2, 0]

In [36]:
fitness(best)

np.float64(2056.72)

In [116]:
item_sets.loc[tmp["set"].dropna()]["name"].value_counts()

name
Set rhoarim            2
Set de Miauvizor       2
Set de Mizz Frizz      1
Set de Gwen Astarde    1
Name: count, dtype: int64

In [115]:
# Print total stats
totals = get_totals(best)
for k,v in totals.items():
    print(k,effects[effects["characteristic"]==k]["name"][0],v)

0 nan -14900.0
1 -#1{{~1~2 a -}}#2 PA 1.0
10 #1{{~1~2 a }}#2 fuerza 560.0
11 #1{{~1~2 a }}#2 vitalidad 1515.0
12 #1{{~1~2 a }}#2 sabiduría 260.0
13 #1{{~1~2 a }}#2 suerte 540.0
14 #1{{~1~2 a }}#2 agilidad 350.0
15 #1{{~1~2 a }}#2 inteligencia 400.0
18 #1{{~1~2 a }}#2 % crítico 77.0
19 -#1{{~1~2 a -}}#2 alcance 3.0
23 -#1{{~1~2 -}}#2 PM 1.0
25 #1{{~1~2 a }}#2 potencia (daños generales) 310.0
26 #1{{~1~2 a }}#2 invocaciones{{~p}}{{~z}} 3.0
27 #1{{~1~2 a }}#2 a la esquiva pérdidas de PA 9.0
33 #1{{~1~2 a }}#2% resistencia a la tierra 15.0
34 #1{{~1~2 a }}#2% resistencia al fuego 30.0
35 #1{{~1~2 a }}#2% resistencia al agua 30.0
36 #1{{~1~2 a }}#2% resistencia al aire 5.0
37 #1{{~1~2 a }}#2% resistencia neutral 5.0
44 #1{{~1~2 a }}#2 iniciativa 1800.0
48 #1{{~1~2 a }}#2 prospección 71.0
49 #1{{~1~2 a }}#2 de cura{{~p}}{{~z}} 10.0
54 #1{{~1~2 a }}#2 de resistencia{{~p}}{{~z}} a la tierra 10.0
78 #1{{~1~2 a }}#2 huida -3.0
79 #1{{~1~2 a }}#2 placaje 23.0
82 #1{{~1~2 a }}#2 al retiro de PA 10

/var/folders/c3/9yq4kl5j7qd715s6mq5tyslm0000gp/T/ipykernel_16848/622476161.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(k,effects[effects["characteristic"]==k]["name"][0],v)


In [117]:
assert False

AssertionError: 

In [130]:
weapon = get_chromosome_weapon(best)
get_damage(weapon,best)

np.float64(2228.7999999999997)

In [121]:
best

[482, 320, 350, 310, 297, 184, 89, 119, 264, 226, 5, 265, 95, 107, 2, 0]

In [76]:
item = tmp.iloc[8]
print(item["name"])
get_item_effects(item)[["name","effectId","diceNum","diceSide"]]

Anillo de Mizz Frizz


,name,effectId,diceNum,diceSide
2861093746004760000,#1{{~1~2 a }}#2 vitalidad,125,251.0,300.0
2861093746004760001,#1{{~1~2 a }}#2 fuerza,118,31.0,50.0
2861093746004760002,#1{{~1~2 a }}#2 inteligencia,126,31.0,50.0
2861093746004760003,#1{{~1~2 a }}#2 suerte,123,31.0,50.0
2861093746004760004,#1{{~1~2 a }}#2 % crítico,115,2.0,3.0
2861093746004760005,#1{{~1~2 a }}#2 de cura{{~p}}{{~z}},178,7.0,10.0
2861093746004760006,#1{{~1~2 a }}#2 de daño{{~p}}{{~z}} neutro,430,7.0,10.0
2861093746004760007,#1{{~1~2 a }}#2 de daño{{~p}}{{~z}} de tierra,422,7.0,10.0
2861093746004760008,#1{{~1~2 a }}#2 de resistencia{{~p}}{{~z}} a l...,240,7.0,10.0
2861093746004760009,-#1{{~1~2 a -}}#2 huida,754,7.0,10.0


In [94]:
item = tmp.iloc[0]
print(item["name"])
get_item_effects(item)[["name","effectId","diceNum","diceSide"]]

Espada maldita del gran Desangrador Guerrero


,name,effectId,diceNum,diceSide
2861093746004750990,#1{{~1~2 a }}#2 de daños de aire,98,9.0,18.0
2861093746004750991,#1{{~1~2 a }}#2 de daños de agua,96,9.0,18.0
2861093746004750992,#1{{~1~2 a }}#2 de daños de fuego,99,9.0,18.0
2861093746004750993,#1{{~1~2 a }}#2 de daños neutrales,100,9.0,18.0
2861093746004750994,#1{{~1~2 a }}#2 de daños de tierra,97,9.0,18.0
2861093746004750995,#1{{~1~2 a }}#2 suerte,123,30.0,0.0
2861093746004750996,#1{{~1~2 a }}#2 inteligencia,126,30.0,0.0
2861093746004750997,#1{{~1~2 a }}#2 fuerza,118,30.0,0.0
2861093746004750998,#1{{~1~2 a }}#2 agilidad,119,30.0,0.0
2861093746004750999,#1{{~1~2 a }}#2 sabiduría,124,30.0,0.0


In [137]:
effects.loc["98"]

name                        #1{{~1~2 a }}#2 de daños de aire
type                                                 Effects
descriptionId                                        1102898
iconId                                                     6
characteristic                                             0
category                                                   2
characteristicOperator                                   NaN
showInTooltip                                              1
useDice                                                    1
forceMinMax                                                1
boost                                                      0
active                                                     0
oppositeId                                                 0
theoreticalDescriptionId                                 NaN
theoreticalPattern                                         1
showInSet                                                  0
bonusType               

In [95]:
contributions[item.name]

characteristic
0     -90.0
10     30.0
11    165.0
12     30.0
13     30.0
14     30.0
15     30.0
33      5.0
34      5.0
35      5.0
36      5.0
37      5.0
Name: effectValue, dtype: float64

In [91]:
item = tmp.iloc[8]
print(item["name"])
get_item_effects(item)[["name","effectId","diceNum","diceSide"]]

Anillo de Mizz Frizz


,name,effectId,diceNum,diceSide
2861093746004760000,#1{{~1~2 a }}#2 vitalidad,125,251.0,300.0
2861093746004760001,#1{{~1~2 a }}#2 fuerza,118,31.0,50.0
2861093746004760002,#1{{~1~2 a }}#2 inteligencia,126,31.0,50.0
2861093746004760003,#1{{~1~2 a }}#2 suerte,123,31.0,50.0
2861093746004760004,#1{{~1~2 a }}#2 % crítico,115,2.0,3.0
2861093746004760005,#1{{~1~2 a }}#2 de cura{{~p}}{{~z}},178,7.0,10.0
2861093746004760006,#1{{~1~2 a }}#2 de daño{{~p}}{{~z}} neutro,430,7.0,10.0
2861093746004760007,#1{{~1~2 a }}#2 de daño{{~p}}{{~z}} de tierra,422,7.0,10.0
2861093746004760008,#1{{~1~2 a }}#2 de resistencia{{~p}}{{~z}} a l...,240,7.0,10.0
2861093746004760009,-#1{{~1~2 a -}}#2 huida,754,7.0,10.0


In [93]:
contributions[item.name]

characteristic
10     50.0
11    300.0
13     50.0
15     50.0
18      3.0
49     10.0
54     10.0
78    -10.0
86     10.0
88     10.0
92     10.0
Name: effectValue, dtype: float64

In [107]:
best

[482, 320, 350, 310, 297, 184, 89, 119, 264, 226, 5, 265, 95, 107, 2, 0]

In [110]:
lst = get_totals(best)
for idx,i in enumerate(lst):
    item = tmp.iloc[idx]
    print(f"{item['name']} (rid:{item.name}):")
    print(i.get("33", 0))  # Assuming "54" is a characteristic ID you want to print

Espada maldita del gran Desangrador Guerrero (rid:2861093746004724544):
5.0
Collar rhoarim (rid:2861093746004741472):
0
Taparrabos rhoarim (rid:2861093746004741473):
0
Botas de Miauvizor (rid:2861093746004731252):
0
Casco de Miauvizor (rid:2861093746004731253):
10.0
Capa Brigada (rid:2861093746004727614):
0
Lokulto (rid:2861093746004728051):
0
Cuadrifolio (rid:2861093746004732259):
0
Anillo de Mizz Frizz (rid:2861093746004728415):
0
Anillo Gwen Astarde (rid:2861093746004726812):
0
Dofus de los hielos (rid:2861093746004723117):
0
Lozano mayor (rid:2861093746004730368):
0
Dolmanax (rid:2861093746004727753):
0
Hércules mayor (rid:2861093746004728123):
0
Dofus Turquesa (rid:2861093746001847037):
0
Dofus Púrpura (rid:2861093746001847005):
0


In [111]:
contributions["2861093746004726812"]

characteristic
10     60.0
12     30.0
13     60.0
18      5.0
26      2.0
44    400.0
48     10.0
82     10.0
84     15.0
88     15.0
90     15.0
92     15.0
Name: effectValue, dtype: float64

In [131]:
item = items.loc["2861093746004730368"]
print(item["name"])
get_item_effects(item)[["name","effectId","diceNum","diceSide"]]
# get_item_contribution(item)

Lozano mayor


,name,effectId,diceNum,diceSide
2861093746004764621,#1{{~1~2 a }}#2 potencia (daños generales),138,80.0,0.0


In [134]:
for i in items["criteria"].unique():
    print(i)

nan
CM<5
CP<12&CM<6&CW>99
CP<11
CP<12
CP<8
cs<25
ci<25
PT!24006&PT!350
PT!368&PT!24029
PT!369&PT!24017
PT!370&PT!24027
Ps=1&Pa>20
CP<9
Ps=2&Pa>30
cs<50
cs<80
cs<100
ca<25
ca<50
ca<80
ca<100
cw<25
cw<50
cw<80
cw<100
cv<25
cv<50
cv<80
cc<25
cv<100
cc<50
cc<80
cc<100
ci<50
ci<80
ci<100
CP<10
Pz=0&Sc!336
CM<6
Ps=1&Pa>10
Pj=44|Pj=48
CC>10
PG=3
CS>99&CA>99&CV>99
PE!8
PJ>2,40|PJ>24,40|PJ>36,40|PJ>28,40|PJ>26,40|PJ>41,40
PJ>2,80|PJ>24,80|PJ>36,80|PJ>28,80|PJ>26,80|PJ>41,80
PJ>2,120|PJ>24,120|PJ>36,120|PJ>28,120|PJ>26,120|PJ>41,120
PJ>2,180|PJ>24,180|PJ>36,180|PJ>28,180|PJ>26,180|PJ>41,180
PZ=1
CI<100&CC<100
BI=1
ci<20
ci<10
cw<20
cw<30
cw<60
ci<30
ci<5
cw<5
ci<60
cw<10
cw<40
ci<70
ci<40
cv<5
cv<30
cv<40
cv<10
cv<20
Ps=2
Ps=1
PN~Silaisie&BI=1
Ps=0
PO!2181&PO!2177&Ps=1&Pa=27&BI=1
PO!2177
PO=2177
PO=2177&Ps=1&Pa=27
CS<40&CI>80
PO!1723
CW>49&Ps=1
CW>49&Ps=2
CA>50
CC>50
CI>90
CS>90
CW>30&CV>50
Po!45
CC>30
CI>299&CC>199
CI>199
PS=0
PS=1
PG=1
PG=2
PG=4
PG=5
PG=6
PG=7
PG=8
PG=9
PG=10
PG=11
PN~Lordsoth

In [132]:
item

name           Lozano mayor
type                  Items
m_flags               24712
id                    16332
nameId               515826
                   ...     
diceNum                 NaN
diceSide                NaN
displayZero             NaN
class                Trofeo
set                     NaN
Name: 2861093746004730368, Length: 69, dtype: object